# Cell 1 - Install Dependencies
Install everything needed for data loading, model training, evaluation, and artifact packaging.

In [ ]:
import sys
import subprocess

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "joblib",
    "sqlalchemy",
    "pymysql",
    "sentence-transformers"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependencies installed successfully")

# Cell 2 - Imports and Runtime Configuration
Import libraries and define constants used across the notebook.

In [ ]:
import json
import os
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR = Path.cwd()
ARTIFACT_DIR = BASE_DIR / "artifacts"
EDA_DIR = BASE_DIR / "eda_outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

CSV_FALLBACK = BASE_DIR / "cmp_training_export.csv"
print(f"Working directory: {BASE_DIR}")

# Cell 3 - Load Data from DB Secrets with CSV Fallback
Try DB first using Colab secrets, then fallback to local CSV export when DB access is not available.

In [ ]:
from sqlalchemy import create_engine

def get_colab_secret(name: str):
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name)
    except Exception:
        return None

def load_from_database() -> pd.DataFrame:
    host = get_colab_secret("DB_HOST")
    port = get_colab_secret("DB_PORT")
    db_name = get_colab_secret("DB_NAME")
    user = get_colab_secret("DB_USER")
    password = get_colab_secret("DB_PASS")

    if not all([host, port, db_name, user, password]):
        raise ValueError("Missing one or more DB secrets in Colab")

    url = f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}"
    engine = create_engine(url)
    query = """
    SELECT
      p.id AS project_id,
      p.workspace_id,
      COALESCE(LOWER(w.org_type), LOWER(o.org_type), 'enterprise') AS org_type,
      p.name AS project_name,
      p.description,
      p.status,
      p.ml_project_type,
      p.ml_complexity,
      p.ml_detected_mode,
      p.ml_domain_tags_json,
      p.ml_constraints_json,
      p.ml_description_embedding,
      p.template_id,
      t.name AS template_name,
      t.template_type,
      t.ml_fitness_score,
      t.ml_completion_rate,
      t.ml_template_embedding,
      pm.user_id,
      pm.role AS member_role,
      pm.ml_assigned_by_ai,
      pm.ml_assignment_confidence,
      wm.ml_role_history_score,
      wm.ml_skill_match_score,
      wm.ml_availability_score,
      wm.ml_chemistry_score,
      wm.ml_top_roles_json
    FROM projects p
    JOIN workspaces w ON w.id = p.workspace_id
    LEFT JOIN organizations o ON o.id = w.organization_id
    LEFT JOIN project_templates t ON t.id = p.template_id
    LEFT JOIN project_members pm ON pm.project_id = p.id AND pm.deleted_at IS NULL
    LEFT JOIN workspace_members wm ON wm.workspace_id = p.workspace_id AND wm.user_id = pm.user_id AND wm.deleted_at IS NULL
    WHERE p.deleted_at IS NULL
    ORDER BY p.id, pm.user_id
    """
    return pd.read_sql_query(query, engine)

def load_training_data() -> pd.DataFrame:
    try:
        df = load_from_database()
        print(f"Loaded {len(df)} rows from database")
        return df
    except Exception as db_exc:
        print(f"DB load unavailable: {db_exc}")
        if not CSV_FALLBACK.exists():
            raise FileNotFoundError(f"CSV fallback not found at {CSV_FALLBACK}")
        df = pd.read_csv(CSV_FALLBACK)
        print(f"Loaded {len(df)} rows from CSV fallback {CSV_FALLBACK}")
        return df

df_raw = load_training_data()
df_raw.head(3)

# Cell 4 - Data Quality Checks and Assertions
Validate minimum data requirements and warn early when dataset quality is weak.

In [ ]:
required_columns = [
    "project_id", "workspace_id", "org_type", "project_name",
    "description", "ml_project_type", "ml_complexity",
    "template_id", "template_name", "template_type",
    "user_id", "member_role", "ml_description_embedding"
]

missing = [c for c in required_columns if c not in df_raw.columns]
assert not missing, f"Missing required columns: {missing}"

unique_projects = df_raw["project_id"].nunique()
unique_templates = df_raw["template_id"].nunique()
unique_members = df_raw["user_id"].nunique()

assert unique_projects >= 30, f"Need at least 30 projects, found {unique_projects}"
assert unique_templates >= 3, f"Need at least 3 templates, found {unique_templates}"
assert unique_members >= 10, f"Need at least 10 members, found {unique_members}"

if unique_projects < 80:
    print("WARNING: Fewer than 80 projects. Stage 3 model may underfit for rare roles.")
if df_raw["member_role"].nunique() < 4:
    print("WARNING: Low role diversity detected. Role classifier confidence may be unstable.")

print("Data checks passed")
print({"projects": unique_projects, "templates": unique_templates, "members": unique_members})

# Cell 5 - Exploratory Data Analysis and PNG Export
Generate quick EDA visuals and save PNG outputs for reporting.

In [ ]:
sns.set_theme(style="whitegrid")

role_counts = df_raw["member_role"].fillna("UNKNOWN").value_counts().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=role_counts.index, y=role_counts.values, palette="Blues_d")
plt.title("Role Distribution")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
role_plot = EDA_DIR / "eda_role_distribution.png"
plt.savefig(role_plot, dpi=150)
plt.show()

template_counts = df_raw["template_type"].fillna("UNKNOWN").value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=template_counts.index, y=template_counts.values, palette="Greens_d")
plt.title("Template Type Usage")
plt.tight_layout()
template_plot = EDA_DIR / "eda_template_usage.png"
plt.savefig(template_plot, dpi=150)
plt.show()

org_counts = df_raw["org_type"].fillna("unknown").value_counts()
plt.figure(figsize=(6, 4))
plt.pie(org_counts.values, labels=org_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Enterprise vs Academic Samples")
org_plot = EDA_DIR / "eda_org_type_split.png"
plt.savefig(org_plot, dpi=150)
plt.show()

print("Saved EDA PNG files:")
for path in [role_plot, template_plot, org_plot]:
    print(" -", path)

# Cell 6 - Stage 3 Model Choice
We use One-vs-Rest Logistic Regression because it trains quickly on small and medium datasets, gives calibrated probabilities, and performs reliably for sparse multi-label role outputs without requiring thousands of examples per label.

# Cell 7 - Train Stage 3 Multi-Label Role Classifier
Train a classifier that predicts required roles per project from semantic and structural features.

In [ ]:
def parse_embedding(value, expected_dim=384):
    if isinstance(value, str) and value.strip():
        try:
            arr = json.loads(value)
            if isinstance(arr, list) and len(arr) >= expected_dim:
                return np.array(arr[:expected_dim], dtype=float)
        except Exception:
            pass
    return np.zeros(expected_dim, dtype=float)

project_level = df_raw.groupby("project_id", as_index=False).agg({
    "ml_description_embedding": "first",
    "ml_project_type": "first",
    "ml_complexity": "first",
    "org_type": "first",
    "template_id": "first"
})

roles_per_project = df_raw.groupby("project_id")["member_role"].apply(lambda s: sorted(set([x for x in s.dropna().astype(str)]))).reset_index()
project_level = project_level.merge(roles_per_project, on="project_id", how="inner")
project_level = project_level[project_level["member_role"].map(len) > 0].copy()

embeddings = np.vstack(project_level["ml_description_embedding"].apply(parse_embedding).values)
ptype_d = pd.get_dummies(project_level["ml_project_type"].fillna("unknown"), prefix="ptype")
complexity_d = pd.get_dummies(project_level["ml_complexity"].fillna("unknown"), prefix="complexity")
org_d = pd.get_dummies(project_level["org_type"].fillna("unknown"), prefix="org")

X_struct = np.hstack([ptype_d.values, complexity_d.values, org_d.values])
X = np.hstack([embeddings, X_struct])

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(project_level["member_role"])

assert y.shape[1] >= 2, "Need at least 2 labels for multi-label training"

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

clf = OneVsRestClassifier(LogisticRegression(max_iter=3000, solver="liblinear"))
clf.fit(X_train, y_train)

y_val_proba = clf.predict_proba(X_val)
y_val_pred_default = (y_val_proba >= 0.5).astype(int)

stage3_metrics = {
    "f1_micro": float(f1_score(y_val, y_val_pred_default, average="micro", zero_division=0)),
    "f1_macro": float(f1_score(y_val, y_val_pred_default, average="macro", zero_division=0)),
    "precision_micro": float(precision_score(y_val, y_val_pred_default, average="micro", zero_division=0)),
    "recall_micro": float(recall_score(y_val, y_val_pred_default, average="micro", zero_division=0)),
}

print("Stage 3 metrics with default threshold 0.50")
print(stage3_metrics)

classifier_payload = {
    "classifier": clf,
    "label_classes": mlb.classes_.tolist(),
    "struct_columns": ptype_d.columns.tolist() + complexity_d.columns.tolist() + org_d.columns.tolist(),
    "embedding_dim": int(embeddings.shape[1]),
}

joblib.dump(classifier_payload, ARTIFACT_DIR / "role_classifier.joblib")
print("Saved role_classifier.joblib")

# Cell 8 - Tune Per-Role Confidence Thresholds
Search thresholds per role to maximize label-specific F1 on validation data.

In [ ]:
thresholds = {}
per_label_f1 = {}
grid = np.arange(0.2, 0.85, 0.05)

for idx, label in enumerate(mlb.classes_):
    best_thr = 0.5
    best_f1 = -1.0
    truth = y_val[:, idx]
    score = y_val_proba[:, idx]

    for thr in grid:
        pred = (score >= thr).astype(int)
        f1 = f1_score(truth, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(round(thr, 2))

    thresholds[label] = best_thr
    per_label_f1[label] = float(best_f1)

with open(ARTIFACT_DIR / "role_thresholds.json", "w", encoding="utf-8") as fp:
    json.dump(thresholds, fp, indent=2)

print("Saved role_thresholds.json")
print(pd.DataFrame({"label": list(thresholds.keys()), "threshold": list(thresholds.values()), "f1": [per_label_f1[k] for k in thresholds.keys()]}))

# Cell 9 - Validate Stage 2 Template Matcher
Use template embeddings and project embeddings to estimate retrieval quality for top-1 template matching.

In [ ]:
template_meta = (
    df_raw[["template_id", "template_name", "template_type", "ml_template_embedding", "ml_fitness_score", "ml_completion_rate"]]
    .dropna(subset=["template_id"])
    .drop_duplicates(subset=["template_id"])
    .copy()
)

if template_meta.empty:
    raise ValueError("No template rows available for Stage 2 validation")

template_vectors = np.vstack(template_meta["ml_template_embedding"].apply(parse_embedding).values)
project_eval = project_level[["project_id", "template_id", "ml_description_embedding"]].copy()
project_vectors = np.vstack(project_eval["ml_description_embedding"].apply(parse_embedding).values)

# Cosine similarity since vectors are approximately normalized
similarity = project_vectors @ template_vectors.T
top_idx = np.argmax(similarity, axis=1)
pred_template_ids = template_meta.iloc[top_idx]["template_id"].values
actual_template_ids = project_eval["template_id"].astype(str).values
stage2_top1 = float(np.mean(pred_template_ids == actual_template_ids))

template_embeddings_npy = ARTIFACT_DIR / "template_embeddings.npy"
np.save(template_embeddings_npy, template_vectors)

template_metadata_csv = ARTIFACT_DIR / "template_metadata.csv"
template_meta.to_csv(template_metadata_csv, index=False)

print(f"Stage 2 top-1 retrieval accuracy: {stage2_top1:.4f}")
print("Saved template_embeddings.npy and template_metadata.csv")

# Cell 10 - Compute Stage 4 Member Profiles
Build and export member profile features used for people matching in FastAPI.

In [ ]:
member_df = (
    df_raw[[
        "workspace_id", "user_id", "member_role",
        "ml_role_history_score", "ml_skill_match_score",
        "ml_availability_score", "ml_chemistry_score",
        "ml_top_roles_json"
    ]]
    .dropna(subset=["workspace_id", "user_id"])
    .copy()
)

for col, default in [
    ("ml_role_history_score", 0.5),
    ("ml_skill_match_score", 0.5),
    ("ml_availability_score", 0.5),
    ("ml_chemistry_score", 0.5),
]:
    member_df[col] = pd.to_numeric(member_df[col], errors="coerce").fillna(default).clip(0.0, 1.0)

profile_rows = (
    member_df.groupby(["workspace_id", "user_id"], as_index=False)
    .agg({
        "ml_role_history_score": "mean",
        "ml_skill_match_score": "mean",
        "ml_availability_score": "mean",
        "ml_chemistry_score": "mean",
        "member_role": lambda s: sorted(set([x for x in s.dropna().astype(str)])),
        "ml_top_roles_json": "first",
    })
)

def safe_roles_json(value, fallback_roles):
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list) and parsed:
                return parsed
        except Exception:
            pass
    return fallback_roles[:3] if fallback_roles else ["DEVELOPER"]

profile_rows["top_roles"] = profile_rows.apply(
    lambda r: safe_roles_json(r["ml_top_roles_json"], r["member_role"]), axis=1
)

profile_rows["profile_vector"] = profile_rows.apply(
    lambda r: [
        round(float(r["ml_role_history_score"]), 6),
        round(float(r["ml_skill_match_score"]), 6),
        round(float(r["ml_availability_score"]), 6),
        round(float(r["ml_chemistry_score"]), 6),
    ],
    axis=1,
)

member_profiles = profile_rows[[
    "workspace_id",
    "user_id",
    "ml_role_history_score",
    "ml_skill_match_score",
    "ml_availability_score",
    "ml_chemistry_score",
    "top_roles",
    "profile_vector",
]].copy()
member_profiles.rename(columns={
    "ml_role_history_score": "roleHistoryScore",
    "ml_skill_match_score": "skillMatchScore",
    "ml_availability_score": "availabilityScore",
    "ml_chemistry_score": "chemistryScore",
}, inplace=True)

member_profiles["top_roles"] = member_profiles["top_roles"].apply(json.dumps)
member_profiles["profile_vector"] = member_profiles["profile_vector"].apply(json.dumps)

member_profiles_path = ARTIFACT_DIR / "member_profiles.csv"
member_profiles.to_csv(member_profiles_path, index=False)
print(f"Saved member profiles to {member_profiles_path} with {len(member_profiles)} rows")

# Cell 11 - Stage Metrics and Remediation Guidance
Print per-stage metrics and provide plain-English guidance when quality is low.

In [ ]:
stage1_proxy = {
    "embedding_coverage": float(np.mean(project_level["ml_description_embedding"].notna())),
    "project_type_coverage": float(np.mean(project_level["ml_project_type"].notna())),
}

stage4_profile_coverage = float(len(member_profiles) / max(df_raw[["workspace_id", "user_id"]].drop_duplicates().shape[0], 1))

metrics_summary = {
    "stage1": stage1_proxy,
    "stage2": {"top1_template_retrieval": stage2_top1},
    "stage3": stage3_metrics,
    "stage4": {"profile_coverage": stage4_profile_coverage},
}

print(json.dumps(metrics_summary, indent=2))

if stage2_top1 < 0.55:
    print("Stage 2 is weak: add more completed projects per template and improve template description quality.")
if stage3_metrics["f1_micro"] < 0.60:
    print("Stage 3 is weak: collect more projects with diverse role combinations, especially minority labels.")
if stage4_profile_coverage < 0.85:
    print("Stage 4 coverage is weak: ensure workspace member ML scores are populated for all active members.")

# Cell 12 - Build Metadata and Artifact Zip
Save metadata, zip exactly required files, and auto-download in Colab.

In [ ]:
from datetime import datetime

model_metadata = {
    "created_at_utc": datetime.utcnow().isoformat() + "Z",
    "embedding_dim": int(embeddings.shape[1]),
    "label_classes": mlb.classes_.tolist(),
    "metrics": metrics_summary,
    "training_rows": int(len(df_raw)),
    "unique_projects": int(df_raw["project_id"].nunique()),
}

model_metadata_path = ARTIFACT_DIR / "model_metadata.json"
with open(model_metadata_path, "w", encoding="utf-8") as fp:
    json.dump(model_metadata, fp, indent=2)

required_files = [
    ARTIFACT_DIR / "role_classifier.joblib",
    ARTIFACT_DIR / "role_thresholds.json",
    ARTIFACT_DIR / "template_embeddings.npy",
    ARTIFACT_DIR / "template_metadata.csv",
    ARTIFACT_DIR / "member_profiles.csv",
    ARTIFACT_DIR / "model_metadata.json",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing artifact file: {file_path}")

zip_path = BASE_DIR / "cmp_pib_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in required_files:
        zf.write(file_path, arcname=file_path.name)

print(f"Artifact zip created: {zip_path}")
print("Zip contains:")
for file_path in required_files:
    print(" -", file_path.name)

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
    print("Auto-download triggered in Colab")
except Exception:
    print("Auto-download skipped (not running in Colab environment)")